# Generation of prompts for generating Dataset

## 1. imports 

In [1]:
import pandas as pd

## 2. Load sample data

In [2]:
sample_df=pd.read_csv("../data/stock_prices/template_gold_silver_var_low_seed7_final.csv")
sample_df

,obs_id,date,t_years,gold,silver,series,price_gold,log_price_gold,dev_gold,sigma_gold,price_silver,log_price_silver,dev_silver,sigma_silver,var_spike,inside_break
0,0,1970-01-01,0.000000,1,1,gold+silver,34.885662,3.552076,0.0,0.01,1.586256,0.461377,0.0,0.018000,0.000000,0
1,1,1970-01-09,0.023810,1,1,gold+silver,35.175087,3.560338,0.0,0.01,1.606812,0.474252,0.0,0.018000,0.000000,0
2,2,1970-01-16,0.043651,1,1,gold+silver,34.071934,3.528474,0.0,0.01,1.541038,0.432456,0.0,0.018000,0.000000,0
3,3,1970-01-26,0.067460,1,1,gold+silver,34.056209,3.528012,0.0,0.01,1.562352,0.446192,0.0,0.018000,0.000000,0
4,4,1970-02-03,0.091270,1,1,gold+silver,34.192275,3.532000,0.0,0.01,1.500128,0.405550,0.0,0.018000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4995,2023-10-27,55.718254,0,1,silver,NaN,NaN,NaN,NaN,12.306814,2.510153,0.0,0.018592,0.000395,1
4996,4996,2023-11-14,55.765873,0,1,silver,NaN,NaN,NaN,NaN,12.586752,2.532645,0.0,0.018450,0.000300,1
4997,4997,2023-11-30,55.813492,0,1,silver,NaN,NaN,NaN,NaN,13.301328,2.587864,0.0,0.018308,0.000205,1
4998,4998,2023-12-18,55.861111,0,1,silver,NaN,NaN,NaN,NaN,12.577272,2.531891,0.0,0.018166,0.000111,1


## 3. Prompt Template

In [ ]:
"""
Prompt template for generating synthetic financial news articles
about gold and silver prices, conditioned on:
  - The article date
  - 21 days of price history ending on that date
  - The commodity of interest (gold or silver)
  - Whether a structural break occurred within the 21-day window
  - If a break occurred: whether the article references it (sampled probabilistically)
  - The current price level of the commodity

Usage:
    from prompt_template import build_prompt
    prompt = build_prompt(date, prices_21d, commodity, break_info, seed)
    # pass prompt to LLM API
"""

import numpy as np
from datetime import date as Date


def build_prompt(
    article_date: str,
    prices_21d: list[float],
    commodity: str,
    current_price: float,
    break_in_window: bool,
    break_date: str | None = None,
    break_magnitude: float | None = None,
    break_direction: str | None = None,
    p_reference_break: float = 0.6,
    seed: int = None,
) -> dict:
    """
    Build a prompt for generating a synthetic financial news article.

    Parameters
    ----------
    article_date      : str   — date of the article, e.g. "1987-10-20"
    prices_21d        : list  — list of 21 daily closing prices ending on article_date
                                (index 0 = oldest, index 20 = article_date)
    commodity         : str   — "gold" or "silver"
    current_price     : float — closing price on article_date (USD/oz)
    break_in_window   : bool  — whether a structural break occurred in the 21-day window
    break_date        : str   — date of the break (if break_in_window=True)
    break_magnitude   : float — log-point magnitude of the break (if break_in_window=True)
    break_direction   : str   — "up" or "down" (if break_in_window=True)
    p_reference_break : float — probability that the article explicitly references
                                the structural break (default 0.6).
                                With prob (1-p), the article covers the same window
                                but frames it as routine market commentary.
    seed              : int   — random seed for the break-reference sampling

    Returns
    -------
    dict with keys:
        "prompt"           : str   — the full prompt to send to the LLM
        "references_break" : bool  — whether the prompt instructs the LLM to
                                     reference the structural break
        "system"           : str   — system message for the LLM
    """
    rng = np.random.default_rng(seed)

    # ── Decide whether the article references the break ───────────────
    references_break = False
    if break_in_window and break_date is not None:
        references_break = bool(rng.random() < p_reference_break)

    # ── Compute summary statistics from price window ──────────────────
    prices = np.array(prices_21d)
    pct_change_21d = (prices[-1] / prices[0] - 1) * 100
    pct_change_5d  = (prices[-1] / prices[-6] - 1) * 100 if len(prices) >= 6 else None
    rolling_vol    = float(np.std(np.diff(np.log(prices))) * np.sqrt(252) * 100)
    price_min      = float(prices.min())
    price_max      = float(prices.max())
    trend          = "upward" if pct_change_21d > 0 else "downward"
    vol_level      = "elevated" if rolling_vol > 20 else "moderate" if rolling_vol > 12 else "low"

    # ── Build the break context block (injected only when relevant) ───
    if break_in_window and references_break and break_date is not None:
        mag_pct = abs(np.exp(break_magnitude) - 1) * 100 if break_magnitude else None
        mag_str = f"{mag_pct:.1f}%" if mag_pct else "significant"
        break_block = f"""
A notable market event occurred on {break_date}: {commodity.capitalize()} prices
experienced a sharp {break_direction}ward move of approximately {mag_str}.
The article should explicitly reference this event. Invent a plausible fictional
cause (e.g. a decision by a fictional central bank, a supply disruption in an
invented mining region, or a policy announcement by a fictional government body).
All institutions, countries, and individuals mentioned must be entirely fictional.
""".strip()
    elif break_in_window and not references_break:
        break_block = f"""
Note: a sharp price move occurred within the observation window, but the article
should NOT directly reference it as a singular event. Instead, frame the price
action as part of broader fictional market dynamics or ongoing trends, using
invented context consistent with the price data.
""".strip()
    else:
        break_block = ""

    # ── System message ─────────────────────────────────────────────────
    system = (
        "You are a financial journalist writing for a commodity markets newswire "
        "in a fictional world. Your articles are 150-250 words, professional in tone, "
        "and written as of the publication date — you only know what has happened "
        "up to and including that date. "
        "Write naturally: mention specific prices and percentage moves where a journalist "
        "would, use qualitative language where a journalist would. "
        "All events, institutions, countries, and individuals must be entirely fictional."
    )

    # ── Main prompt ────────────────────────────────────────────────────
    prompt = f"""Write a short financial news article about the {commodity} market,
published on {article_date}. This takes place in a fictional world — all events,
institutions, countries, and named individuals must be invented.

CONTEXT (use this to inform the article — do not copy it verbatim):
  Commodity      : {commodity}
  Current price  : ${current_price:.2f}/oz  (as of {article_date})
  21-day trend   : {trend} ({pct_change_21d:+.2f}% over the window)
  5-day trend    : {f'{pct_change_5d:+.2f}%' if pct_change_5d is not None else 'n/a'}
  Price range    : ${price_min:.2f} - ${price_max:.2f}/oz over the past 21 days
  Volatility     : {vol_level} ({rolling_vol:.1f}% annualized)
{('  Recent event  : sharp ' + (break_direction or '') + 'ward price move on ' + (break_date or '')) if break_in_window and references_break else ''}

{break_block}

INSTRUCTIONS:
- Write as of {article_date}: only reference events up to this date.
- You may mention specific prices, percentage moves, or price ranges where
  natural — or use qualitative language ("surged", "edged lower", "remained
  range-bound") — exactly as a journalist would decide.
- Invent plausible fictional causes for the price movements consistent with
  the context above (fictional central banks, invented mining regions, made-up
  trade bodies, fictional geopolitical tensions).
- Do NOT use future tense or make price predictions.
- Keep the article between 150 and 250 words.
- Start with the headline, then the body. No preamble.
"""

    return {
        "system":           system,
        "prompt":           prompt.strip(),
        "references_break": references_break,
    }


if __name__ == "__main__":
    # ── Example usage ────────────────────────────────────────────────
    import json

    example_prices = [
        1050.0, 1055.2, 1048.3, 1062.1, 1071.5,
        1068.9, 1060.4, 1055.0, 1049.8, 1044.2,
        1038.5, 1030.1, 1025.7, 1019.3, 1012.0,
         985.4,  960.1,  941.3,  938.7,  945.2,  952.8
    ]

    result = build_prompt(
        article_date      = "2008-10-15",
        prices_21d        = example_prices,
        commodity         = "gold",
        current_price     = 952.8,
        break_in_window   = True,
        break_date        = "2008-10-06",
        break_magnitude   = -0.28,
        break_direction   = "down",
        p_reference_break = 0.6,
        seed              = 42,
    )

    print("=" * 60)
    print(f"References break: {result['references_break']}")
    print("=" * 60)
    print("SYSTEM:")
    print(result["system"])
    print("\nPROMPT:")
    print(result["prompt"])